In [ ]:
import os
import glob

# 라벨 폴더 내부의 모든 .cache 파일 찾아 삭제
cache_files = glob.glob('/content/drive/MyDrive/posco_vision/datasets/labels/**/*.cache', recursive=True)
for cf in cache_files:
    os.remove(cf)
    print(f"🗑️ 캐시 파일 삭제 완료: {cf}")

if not cache_files:
    print("🧹 삭제할 캐시 파일이 없습니다 (클린 상태).")

# 2 stage 분리
- 부품 검출
- 결합 검출

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install ultralytics

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.1 MB/s eta 0:00:00


In [ ]:
# !pip install "sympy<1.13" --force-reinstall
# 1. 기존의 충돌 패키지를 최신 안정 버젼으로 강제 재설치
# # !pip install --upgrade --force-reinstall torch torchvision ultralytics
# !pip install ultralytics

### 1. 변환 코드

데이터셋 분리

In [ ]:
import os
import json
import glob
import shutil
import random
from PIL import Image

IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/posco_vision'
else:
    notebook_path = globals().get("__vsc_ipynb_file__")
    if notebook_path:
        BASE_DIR = os.path.dirname(os.path.abspath(notebook_path))
    elif "__file__" in globals():
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    else:
        BASE_DIR = os.getcwd()
        while not os.path.isdir(os.path.join(BASE_DIR, "datasets")):
            parent_dir = os.path.dirname(BASE_DIR)
            if parent_dir == BASE_DIR:
                break
            BASE_DIR = parent_dir

DATASETS_DIR = os.path.join(BASE_DIR, "datasets")
JSON_RAW_DIR = os.path.join(DATASETS_DIR, "json_raw")
if not os.path.isdir(JSON_RAW_DIR):
    raise FileNotFoundError(f"JSON dataset directory not found: {JSON_RAW_DIR}")

# Stage별 클래스 매핑 (0부터 시작)
PART_CLASS_MAPPING = {
    "body": 0, "lead": 1, "tab": 2
}

DEFECT_CLASS_MAPPING = {
    "scratch": 0, "crash": 1, "bend": 2
}

def create_stage_directories(stage_name):
    stage_dir = os.path.join(DATASETS_DIR, f"dataset_{stage_name}")
    img_train = os.path.join(stage_dir, "images", "train")
    img_val = os.path.join(stage_dir, "images", "val")
    txt_train = os.path.join(stage_dir, "labels", "train")
    txt_val = os.path.join(stage_dir, "labels", "val")

    # 기존 폴더 초기화 (잔재 파일 제거)
    if os.path.exists(stage_dir):
        shutil.rmtree(stage_dir)

    for path in [img_train, img_val, txt_train, txt_val]:
        os.makedirs(path, exist_ok=True)

    return stage_dir, img_train, img_val, txt_train, txt_val

def convert_labelme_json(json_path, img_w, img_h, class_mapping):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    yolo_lines = []
    for shape in data.get('shapes', []):
        label = shape.get('label')
        if label not in class_mapping:
            continue

        class_id = class_mapping[label]
        points = shape.get('points', [])

        if len(points) < 3:
            continue

        normalized_points = []
        for p in points:
            norm_x = min(max(p[0] / img_w, 0.0), 1.0)
            norm_y = min(max(p[1] / img_h, 0.0), 1.0)
            normalized_points.append(f"{norm_x:.6f} {norm_y:.6f}")

        points_str = " ".join(normalized_points)
        yolo_lines.append(f"{class_id} {points_str}")

    return yolo_lines

def create_yaml_file(stage_dir, stage_type, class_mapping):
    names_dict_lines = [
        f"  {cls_id}: '{name}'"
        for name, cls_id in sorted(class_mapping.items(), key=lambda x: x[1])
    ]
    names_str = "\n".join(names_dict_lines)

    yaml_content = f"""path: {stage_dir}
train: images/train
val: images/val

names:
{names_str}
"""
    yaml_path = os.path.join(DATASETS_DIR, f"data_{stage_type}.yaml")
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(yaml_content)
    print(f"📄 [{stage_type.upper()}] yaml 생성 완료: {yaml_path}")

def split_files_by_keyword(json_files, val_ratio=0.2):
    """ 파일명의 양품/불량 키워드를 기반으로 Train/Val 분할 """
    normal_files = []
    defect_files = []

    for f in json_files:
        filename = os.path.basename(f).lower()
        if 'normal' in filename:
            normal_files.append(f)
        else:
            defect_files.append(f)

    random.seed(42)
    random.shuffle(normal_files)
    random.shuffle(defect_files)

    # 각각 8:2 비율로 분할
    norm_val_cnt = max(1, int(len(normal_files) * val_ratio)) if normal_files else 0
    def_val_cnt = max(1, int(len(defect_files) * val_ratio)) if defect_files else 0

    val_files = set(normal_files[:norm_val_cnt] + defect_files[:def_val_cnt])

    print(f"📊 [데이터 분할 결과]")
    print(f"  - 정상(normal) 파일: 총 {len(normal_files)}개 (Train: {len(normal_files)-norm_val_cnt}, Val: {norm_val_cnt})")
    print(f"  - 불량 파일: 총 {len(defect_files)}개 (Train: {len(defect_files)-def_val_cnt}, Val: {def_val_cnt})")

    return val_files

def process_2stage_dataset(val_ratio=0.2):
    json_files = glob.glob(os.path.join(JSON_RAW_DIR, "*.json"))
    if not json_files:
        print(f"❌ 오류: '{JSON_RAW_DIR}' 디렉토리에서 .json 파일을 찾지 못했습니다!")
        return

    val_files = split_files_by_keyword(json_files, val_ratio)

    part_dir, p_img_train, p_img_val, p_txt_train, p_txt_val = create_stage_directories("part")
    defect_dir, d_img_train, d_img_val, d_txt_train, d_txt_val = create_stage_directories("defect")

    print(f"\n🚀 [환경: {'COLAB' if IS_COLAB else 'LOCAL'}] 변환 작업 시작...\n")

    part_count_total, defect_count_total = 0, 0

    for json_file in json_files:
        base_name = os.path.splitext(os.path.basename(json_file))[0]

        img_file = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG', '.JPEG']:
            possible_path = os.path.join(JSON_RAW_DIR, base_name + ext)
            if os.path.exists(possible_path):
                img_file = possible_path
                break

        if not img_file:
            continue

        try:
            with Image.open(img_file) as img:
                img_w, img_h = img.size
        except Exception as e:
            print(f"⚠️ 경고: {img_file} 읽기 실패 - {e}")
            continue

        part_yolo_lines = convert_labelme_json(json_file, img_w, img_h, PART_CLASS_MAPPING)
        defect_yolo_lines = convert_labelme_json(json_file, img_w, img_h, DEFECT_CLASS_MAPPING)

        is_val = json_file in val_files

        # 1. Part Dataset 저장 (모든 이미지 포함)
        target_p_img = p_img_val if is_val else p_img_train
        target_p_txt = p_txt_val if is_val else p_txt_train
        shutil.copy(img_file, os.path.join(target_p_img, os.path.basename(img_file)))
        with open(os.path.join(target_p_txt, base_name + ".txt"), 'w', encoding='utf-8') as f:
            f.write("\n".join(part_yolo_lines))
        part_count_total += len(part_yolo_lines)

        # 2. Defect Dataset 저장 (양품은 빈 txt로 저장되어 오탐 방지용 네거티브 샘플로 학습)
        target_d_img = d_img_val if is_val else d_img_train
        target_d_txt = d_txt_val if is_val else d_txt_train
        shutil.copy(img_file, os.path.join(target_d_img, os.path.basename(img_file)))
        with open(os.path.join(target_d_txt, base_name + ".txt"), 'w', encoding='utf-8') as f:
            f.write("\n".join(defect_yolo_lines))
        defect_count_total += len(defect_yolo_lines)

    create_yaml_file(part_dir, "part", PART_CLASS_MAPPING)
    create_yaml_file(defect_dir, "defect", DEFECT_CLASS_MAPPING)

    print("\n🎉 2-Stage 데이터 변환 완료!")
    print(f"  - [Part]   총 라벨 객체 수: {part_count_total}개")
    print(f"  - [Defect] 총 라벨 객체 수: {defect_count_total}개")

if __name__ == "__main__":
    process_2stage_dataset(val_ratio=0.2)

### 2. 학습코드

In [ ]:
import os
import shutil
from ultralytics import YOLO

IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # 💥 바로가기 경로 대신 내 드라이브의 실제 마운트 경로 사용
    DRIVE_SAVE_DIR = '/content/drive/MyDrive/posco_vision/YOLO_results'
    # 💥 학습은 코랩 초고속 로컬 SSD 디스크에서 진행!
    LOCAL_PROJECT_PATH = '/content/YOLO_results'
else:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    LOCAL_PROJECT_PATH = os.path.join(BASE_DIR, "YOLO_results")
    DRIVE_SAVE_DIR = LOCAL_PROJECT_PATH

def train_and_sync_model(stage_type='defect', model_version='v4-hsv-93011'):
    yaml_path = f"/content/drive/MyDrive/posco_vision/datasets/data_{stage_type}.yaml" if IS_COLAB else f"./datasets/data_{stage_type}.yaml"
    model_name = f"mosfet_{stage_type}_{model_version}"

    print(f"🚀 [{stage_type.upper()} 모델] 로컬 SSD('{LOCAL_PROJECT_PATH}')에서 학습 시작...")

    model = YOLO("yolov8m-seg.pt")

    results = model.train(
        data=yaml_path,
        epochs=150,
        imgsz=800,
        batch=8,
        device='cpu', # 0,
        patience=50,
        overlap_mask=True,
        cls=1.5,                 # 결함 클래스 가중치 강화
        box=7.5,
        scale=0.3,
        mosaic=0.0 if stage_type == 'defect' else 1.0, # 결함 모델 학습 시 Mosaic 끔 (오탐 방지)
        degrees=10.0,
        fliplr=0.5,
        flipud=0.0,
        exist_ok=True,
        project=LOCAL_PROJECT_PATH,
        name=model_name,
    )

    # ---------------------------------------------------------
    # 💥 학습 완료 후 구글 드라이브로 안정적 동기화 (Flush & Sync)
    # ---------------------------------------------------------
    if IS_COLAB:
        local_weight_dir = os.path.join(LOCAL_PROJECT_PATH, model_name, "weights")
        drive_weight_dir = os.path.join(DRIVE_SAVE_DIR, model_name, "weights")

        os.makedirs(drive_weight_dir, exist_ok=True)

        print("\n📂 구글 드라이브로 가중치 파일 안심 복사 중...")
        for pt_file in ["best.pt", "last.pt"]:
            src = os.path.join(local_weight_dir, pt_file)
            dst = os.path.join(drive_weight_dir, pt_file)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"  ✅ 복사 완료: {pt_file} -> {dst}")

        # 구글 드라이브 강제 캐시 비우기 (Sync)
        os.system("sync")
        print("🎉 구글 드라이브 동기화 완료!")

if __name__ == "__main__":
    train_and_sync_model(stage_type='defect', model_version='v0')

🚀 [DEFECT 모델] 로컬 SSD('/content/YOLO_results')에서 학습 시작...
Ultralytics 8.4.166 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/posco_vision/datasets/data_defect.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.93

### 3. 추론

### 모델 추론

In [1]:
import os
import glob
from datetime import datetime
from PIL import Image
import cv2
from ultralytics import YOLO

# ---------------------------------------------------------
# [환경 설정 플래그] 코랩: True / 로컬: False
# ---------------------------------------------------------
IS_COLAB = True

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/posco_vision'
    INFER_DEVICE = 'cpu'  # 코랩 무료 플랜 추론 시 CPU 권장 (GPU 필요 시 0)
else:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    INFER_DEVICE = 0      # 로컬 GPU 환경일 경우 0, 없으면 'cpu'

PROJECT_PATH = os.path.join(BASE_DIR, "YOLO_results")

PART_CLASS_NAMES = {
    0: "body", 1: "lead", 2: "tab", 3: "text_model", 4: "text_lot", 5: "text_id"
}

DEFECT_CLASS_NAMES = {
    0: "crash", 1: "scratch", 2: "bend"
}

def get_weight_path(model_name):
    best_path = os.path.join(PROJECT_PATH, model_name, "weights", "best.pt")
    last_path = os.path.join(PROJECT_PATH, model_name, "weights", "last.pt")

    if os.path.exists(best_path):
        return best_path
    elif os.path.exists(last_path):
        print(f"⚠️ '{model_name}'의 best.pt가 없어 last.pt를 사용합니다.")
        return last_path
    else:
        raise FileNotFoundError(f"❌ 가중치 파일을 찾을 수 없습니다: {best_path}")

def run_2stage_inference(part_model_name, defect_model_name, custom_img_path=None):
    part_weight = get_weight_path(part_model_name)
    defect_weight = get_weight_path(defect_model_name)

    print(f"🔍 [Stage 1] 부품 모델 가중치: {part_weight}")
    print(f"🔍 [Stage 2] 결함 모델 가중치: {defect_weight}\n")

    part_model = YOLO(part_weight)
    defect_model = YOLO(defect_weight)

    test_images = []
    if custom_img_path:
        if os.path.exists(custom_img_path):
            test_images.append(custom_img_path)
        else:
            print(f"❌ 지정한 이미지 파일이 없습니다: {custom_img_path}")
            return
    else:
        test_dir = os.path.join(BASE_DIR, "datasets", "test_raw")
        os.makedirs(test_dir, exist_ok=True)
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG', '*.JPEG']:
            test_images.extend(glob.glob(os.path.join(test_dir, ext)))

        if not test_images:
            print(f"⚠️ 'datasets/test_raw' 폴더에 테스트할 이미지가 없습니다: {test_dir}")
            return

    save_dir = os.path.join(BASE_DIR, "inference_results_2stage")
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print(f"🚀 [환경: {'COLAB' if IS_COLAB else 'LOCAL'}] 총 {len(test_images)}개 이미지 추론 진행 중...\n")

    for img_path in test_images:
        file_name = os.path.basename(img_path)
        name_only, ext_only = os.path.splitext(file_name)

        try:
            with Image.open(img_path) as pil_img:
                part_res = part_model.predict(source=pil_img, conf=0.35, iou=0.45, device=INFER_DEVICE)[0]
                defect_res = defect_model.predict(source=pil_img, conf=0.20, iou=0.35, device=INFER_DEVICE)[0]
        except Exception as e:
            print(f"❌ 추론 실패 ({file_name}): {e}")
            continue

        print(f"📸 [결과 요약] 파일명: {file_name}")
        print("  🔹 [부품 탐지]")
        for box in part_res.boxes:
            cls_id, conf = int(box.cls[0].item()), float(box.conf[0].item())
            print(f"     - [{PART_CLASS_NAMES.get(cls_id, cls_id)}] {conf:.2f}")

        print("  🔴 [결함 탐지]")
        for box in defect_res.boxes:
            cls_id, conf = int(box.cls[0].item()), float(box.conf[0].item())
            print(f"     - ⚠️ [{DEFECT_CLASS_NAMES.get(cls_id, cls_id)}] {conf:.2f}")

        # 두 모델 오버레이 시각화 병합
        img_part_bgr = part_res.plot()
        img_defect_bgr = defect_res.plot()
        combined_bgr = cv2.addWeighted(img_part_bgr, 0.5, img_defect_bgr, 0.5, 0)
        res_pil = Image.fromarray(combined_bgr[..., ::-1])

        save_path = os.path.join(save_dir, f"res2s_{timestamp}_{name_only}{ext_only}")
        res_pil.save(save_path)
        print(f"  💾 저장 완료: {save_path}\n")

    print(f"🎉 추론 완료! 저장 위치: '{save_dir}'")

if __name__ == "__main__":
    run_2stage_inference(
        part_model_name='mosfet_part_v4-hsv-93012',
        defect_model_name='mosfet_defect_v4-hsv-93012'
    )

ModuleNotFoundError: No module named 'ultralytics'

#### 데이터 테스트 스크립트

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. 구글 드라이브 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. 프로젝트 기본 경로 설정
# /content/drive/MyDrive/posco_vision/datasets/images/train
BASE_DIR = Path('/content/drive/MyDrive/posco_vision/datasets')

# 점검할 데이터셋 및 클래스 이름 정의
DATASETS_TO_CHECK = {
    "부품 데이터셋 (Part)": {
        "dir": BASE_DIR / "dataset_part" / "labels" / "train",
        "names": {0: "body", 1: "lead", 2: "tab", 3: "text_model", 4: "text_lot", 5: "text_id"}
    },
    "결함 데이터셋 (Defect)": {
        "dir": BASE_DIR / "dataset_defect" / "labels" / "train",
        "names": {0: "crash", 1: "scratch", 2: "bend"}
    }
}

def inspect_dataset(name, config):
    label_dir = config["dir"]
    class_names = config["names"]

    print(f"==========================================")
    print(f"🔍 [{name}] 라벨 점검 시작")
    print(f"📁 경로: {label_dir}")
    print(f"==========================================")

    if not label_dir.exists():
        print(f"❌ 오류: 해당 경로가 존재하지 않습니다!\n")
        return

    txt_files = list(label_dir.glob("*.txt"))
    total_files = len(txt_files)
    print(f"📄 총 라벨(.txt) 파일 개수: {total_files}개")

    if total_files == 0:
        print(f"⚠️ 경고: 라벨 파일이 하나도 없습니다.\n")
        return

    empty_files = []
    class_counts = {}
    invalid_class_ids = []

    for txt_path in txt_files:
        with open(txt_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]

            if not lines:
                empty_files.append(txt_path.name)
                continue

            for line in lines:
                parts = line.split()
                cls_id = int(parts[0])

                # 정해진 클래스 범위를 벗어난 ID 체크
                if cls_id not in class_names:
                    invalid_class_ids.append((txt_path.name, cls_id))

                class_counts[cls_id] = class_counts.get(cls_id, 0) + 1

    # 결과 출력
    print(f"⚠️ 빈 라벨 파일 수 (객체 미검출/라벨 누락): {len(empty_files)}개 / {total_files}개")
    if empty_files:
        print(f"   └ 빈 파일 예시 (최대 5개): {empty_files[:5]}")

    print("\n📊 클래스 ID별 객체(Polygon/Box) 개수:")
    for cls_id in sorted(class_names.keys()):
        cls_name = class_names[cls_id]
        count = class_counts.get(cls_id, 0)
        print(f"   - Class {cls_id} ({cls_name}): {count}개")

    if invalid_class_ids:
        print(f"\n🚨 [경고] 허용되지 않은 Class ID 발견 ({len(invalid_class_ids)}개):")
        for file_name, bad_id in invalid_class_ids[:5]:
            print(f"   - {file_name}: Class ID {bad_id}")
    else:
        print("\n✅ 클래스 ID 범주 정상 (0번부터 연속적으로 잘 매핑됨)")

    print("\n")

# 실행
for name, config in DATASETS_TO_CHECK.items():
    inspect_dataset(name, config)